In [1]:
#import spacy
from jiwer import compute_measures
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
from jiwer import RemovePunctuation

#SPACY_MODEL = "en_core_web_sm"
#nlp = spacy.load(SPACY_MODEL)

def identify_edit_type(ref, asr, llm):
    edit_types = []
    edits = {"A":[],"B":[],"C":[],"D":[]}
    for r, a, l in zip(ref, asr, llm):
        if a == l == r:
            edit_types.append("C")  # left it correct
            edits["C"].append((a,l,r))
        elif a != l and l == r:
            edit_types.append("A")  # improve it
            edits["A"].append((a,l,r))
        elif a != r and l != r:
            edit_types.append("D")  # left it incorrect
            edits["D"].append((a,l,r))
        elif a == r and l != r:
            edit_types.append("B")  # introducing an error
            edits["B"].append((a,l,r))
    return edit_types, edits


In [2]:
# Load your data
# Example:
path = "/home/mnaderi/Documents/thesis/chat-gpt-asr/results/results-dev-set/results_noisy/results_tiny/results_sentence_confidence_tiny/results_GPT-3.5-Turbo_tiny/gpt-3.5-turbo-0125/results_without_sentence_confidence_tiny/corrected_transcriptions_sentence_confidence_tiny.json"

with open(path, "r") as f:
    data = json.load(f)
    
    transcriptions = [RemovePunctuation()(d["asr_transcription"]["text"]).lower().strip() for d in data]
    reference_transcriptions = [RemovePunctuation()(d["reference_transcription"]).lower().strip() for d in data]
    corrected_transcriptions = [RemovePunctuation()(d["corrected_asr_transcription"]).lower().strip() for d in data]

In [3]:
#Compute edit types
for i in range(len(data))[2000:2010]:
    ref= reference_transcriptions[i].split()
    llm= corrected_transcriptions[i].split()
    asr= transcriptions[i].split()
    edit_types, edits = identify_edit_type(ref, asr, llm)
    word_counts = {'A': 0, 'B': 0, 'C': 0, 'D': 0}
    for edit_type in edit_types:
        word_counts[edit_type] += 1
    
    # Create a DataFrame to store word counts by edit type
    df_edit_types = pd.DataFrame(word_counts.items(), columns=['Type', 'Count'])
    # df_edit_types.set_index('Type', inplace=True)
    
    # Display the DataFrame
    print("asr: {} len:{} \nllm: {} len: {} \nref: {} len: {}".format(asr, len(asr), llm, len(llm), ref, len(ref)))
    print('edits: ', edits)
    print(i, df_edit_types , "\n")

asr: ['his', 'body', 'was', 'long', 'and', 'slender', 'hard', 'and', 'agile', 'his', 'sight', 'keen', 'his', 'aim', 'and', 'airing'] len:16 
llm: ['his', 'body', 'was', 'long', 'and', 'slender', 'hard', 'and', 'agile', 'his', 'sight', 'keen', 'his', 'aim', 'unerring'] len: 15 
ref: ['his', 'body', 'was', 'long', 'and', 'slender', 'hard', 'and', 'agile', 'his', 'sight', 'keen', 'his', 'aim', 'unerring'] len: 15
edits:  {'A': [('and', 'unerring', 'unerring')], 'B': [], 'C': [('his', 'his', 'his'), ('body', 'body', 'body'), ('was', 'was', 'was'), ('long', 'long', 'long'), ('and', 'and', 'and'), ('slender', 'slender', 'slender'), ('hard', 'hard', 'hard'), ('and', 'and', 'and'), ('agile', 'agile', 'agile'), ('his', 'his', 'his'), ('sight', 'sight', 'sight'), ('keen', 'keen', 'keen'), ('his', 'his', 'his'), ('aim', 'aim', 'aim')], 'D': []}
2000   Type  Count
0    A      1
1    B      0
2    C     14
3    D      0 

asr: ['one', 'morning', 'as', 'gandhi', 'was', 'seated', 'in', 'his', 'boat',

In [4]:
# debug
products = ["salt","cloth","toothbrush","Maryam"]
prices = [1.95, 100, 5]

list(zip(products,prices))

[('salt', 1.95), ('cloth', 100), ('toothbrush', 5)]